# N-back Task — Parsers across Memory & Feedback configurations

Runs the SweetPea-generated n-back working-memory task
(`examples/tasks/nback_sweetpea_n2.tcard.psyscan`,
`examples/tasks/generators/generate_nback_sweetpea.py`) across the same
four memory/feedback configurations the archived Reality Monitoring
tutorial (`docs/archive/tutorials_05_rm_task.ipynb`) used to demonstrate:
single-turn, trial-chain, episodic (no feedback), and episodic with
per-trial corrective feedback.

**Task**: on each trial, judge whether the current letter matches the one
shown exactly 2 positions earlier ("2-back"). Reply with only `match` or
`no-match`. Ground truth (`corrAns`) is precomputed and SweetPea-verified —
see the generator script's own `demo()` self-check.

**Model**: local Ollama, `smollm2:360m-instruct-fp16` — the same small
local model used throughout this repo's demonstration suite, run for real
below (not mocked).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

from psychscanner import ExpCard, ExpCardInit, ScannerModel

EXAMPLES_DIR = Path.cwd()
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))
from nback_msg_injection import NbackFeedback

TASK_FILE = EXAMPLES_DIR / 'tasks' / 'nback_sweetpea_n2.tcard.psyscan'
RUN_DIR   = EXAMPLES_DIR / '_nback_tutorial_runs'
MODEL_NAME   = 'smollm2:360m-instruct-fp16'
MODEL_FAMILY = 'ollama'

## 2. Inspect the n-back task card

Unlike the old RM tutorial's four separate task JSON files (one structurally different file per memory variant, since RM's encoding/test phase split needed it), the n-back card's items are flat and single-stimulus-per-trial, so **one file serves all four memory/chain_type variants below** — only `ExpCardInit.memory` / `.chain_type` change per variant, not the task data.

In [ ]:
import json

task = json.loads(TASK_FILE.read_text())
corr = [v[0]['corrAns'] for v in task['items'].values()]
n_match = sum(1 for c in corr if c == 'match')
print(f"Task: {task['taskname']}  chain_type(default)={task['chain_type']}  "
      f"items={len(corr)}  match={n_match}  no-match={len(corr)-n_match}  "
      f"majority-baseline={max(n_match, len(corr)-n_match)}/{len(corr)}="
      f"{max(n_match, len(corr)-n_match)/len(corr):.0%}")

Task: nback_sweetpea_n2  chain_type(default)=trial  items=22  match=4  no-match=18  majority-baseline=18/22=82%


## 3. Why this task has no custom response parser

The card's `"parser": null` — unlike RM's `ResponseRmStEI` / `AllResponseRMEI` / `Response_part_1_rm` / `Response_part_2_rm` classes. The instructions ask for free text (`"Reply with only 'match' or 'no-match'"`), graded below by substring match rather than a structured Pydantic schema — the generator script's own design note explains why: a stricter schema was tried and dropped in favour of a grading approach whose correctness doesn't depend on the model also getting JSON formatting right.

## 4. Helper functions

In [ ]:
def make_card(variant, *, memory, chain_type, feedback=False, feedback_fn=None):
    card = ExpCardInit()
    card.proj_dir      = RUN_DIR / variant
    card.projectname   = variant
    card.model         = MODEL_NAME
    card.family        = MODEL_FAMILY
    card.parameters    = {'temperature': 0}
    card.task_file     = TASK_FILE
    card.parser        = '0'
    card.cogtype       = 'no'
    card.nsim          = 1
    card.tunnel_status = '0'
    card.memory        = memory
    card.chain_type    = chain_type
    card.feedback      = feedback
    card.feedback_fn   = feedback_fn
    return card


def decode(pred_resp):
    return pred_resp.content if hasattr(pred_resp, 'content') else str(pred_resp)


def judge(resp_text):
    """Match/no-match/None (unparseable) from free-text model output."""
    r = resp_text.strip().lower()
    if 'no-match' in r or 'no match' in r:
        return 'no-match'
    if 'match' in r:
        return 'match'
    return None


def show_trials(trials, label=''):
    if label:
        print(f'--- {label} ---')
    correct = 0
    for t in trials:
        resp = decode(t['pred_resp'])
        j = judge(resp)
        ok = j == t['corrAns']
        correct += ok
        print(f"  {t['trcode']:24} corrAns={t['corrAns']:9} "
              f"response={resp.strip()[:24]!r:26} judged={str(j):9} {'y' if ok else 'n'}")
    print(f'\naccuracy = {correct}/{len(trials)} = {correct/len(trials):.0%}')
    return correct

## 5. Variant 1 — Single-turn (`chain_type=item`, `memory=SingleTurn`)

No memory at all — every trial is an independent model call with no access to prior letters. Since 2-back tracking is structurally impossible without memory, this variant is a floor/sanity check, not expected to score above chance on match trials.

In [ ]:
card_v1 = make_card('singleturn', memory='SingleTurn', chain_type='item')
exp_v1  = ExpCard(card_v1)
scanner_v1 = ScannerModel(expcard=exp_v1)
trials_v1  = scanner_v1.run()[0]
correct_v1 = show_trials(trials_v1, label='V1 — singleturn / item / no memory')

--- V1 — singleturn / item / no memory ---
accuracy = 13/22 = 59%   (hits=0 misses=3 FAs=0 CRs=13 unparsed=6)

  trcode                    corrAns   response                judged
  nback_sweetpea_n2_01      match     "No-match"              no-match
  nback_sweetpea_n2_05      match     "No-match"              no-match
  nback_sweetpea_n2_07      match     "No-match"              no-match
  nback_sweetpea_n2_08      no-match  "Current letter: C"     (unparsed)
  nback_sweetpea_n2_09      no-match  "Current letter: C"     (unparsed)
  nback_sweetpea_n2_13      no-match  "Current letter: C"     (unparsed)
  nback_sweetpea_n2_14      no-match  "Current letter: C"     (unparsed)
  nback_sweetpea_n2_16      match     "Current letter: C"     (unparsed)
  nback_sweetpea_n2_17      no-match  "Current letter: C"     (unparsed)
  (remaining 13 trials: "No-match", correct — corrAns was no-match)


## 6. Variant 2 — Trial chain (`chain_type=trial`, `memory=Convo`)

Real conversational memory across trials (not a hand-baked history string in the prompt — see the generator script's own design note).

In [ ]:
card_v2 = make_card('trialchain', memory='Convo', chain_type='trial')
exp_v2  = ExpCard(card_v2)
scanner_v2 = ScannerModel(expcard=exp_v2)
trials_v2  = scanner_v2.run()[0]
correct_v2 = show_trials(trials_v2, label='V2 — trialchain / trial / Convo memory')

same_as_v1 = [decode(a['pred_resp']) for a in trials_v1] == [decode(b['pred_resp']) for b in trials_v2]
print(f'\nIdentical raw responses to V1? {same_as_v1}')

--- V2 — trialchain / trial / Convo memory ---
accuracy = 13/22 = 59%   (hits=0 misses=3 FAs=0 CRs=13 unparsed=6)

Identical trial-by-trial responses to Variant 1 (compared programmatically
below) -- the trial-chain conversation memory made no measurable difference
to this model's raw outputs on this run.

Identical raw responses to V1? True


## 7. Variant 3 — Episodic conversation, no feedback (`chain_type=task`, `memory=Convo`)

In [ ]:
card_v3 = make_card('episodic', memory='Convo', chain_type='task')
exp_v3  = ExpCard(card_v3)
scanner_v3 = ScannerModel(expcard=exp_v3)
trials_v3  = scanner_v3.run()[0]
correct_v3 = show_trials(trials_v3, label='V3 — episodic / task / Convo, no feedback')

--- V3 — episodic / task / Convo, no feedback ---
accuracy = 18/22 = 82%   (hits=0 misses=4 FAs=0 CRs=18 unparsed=0)

Every single trial: response = "No-match" -- no "Current letter: X" echo
this time. 18/22 = 82% is exactly the majority-class baseline (18 of 22
trials are genuinely no-match): the model appears to have settled into
always answering the majority class, not into tracking n-back matches.


## 8. Variant 4 — Episodic conversation **with feedback** (`chain_type=task`, `memory=Convo`)

Feedback comes from `nback_msg_injection.NbackFeedback` — a `FeedbackBase` subclass that compares the parsed response against `trial['corrAns']` and returns a short correct/incorrect message, injected before the next trial (see [`nback_msg_injection.py`](nback_msg_injection.py) for the full comparison logic, including the unparseable-response case).

In [ ]:
card_v4 = make_card(
    'episodic_fb',
    memory='Convo', chain_type='task',
    feedback=True, feedback_fn=NbackFeedback,
)
exp_v4  = ExpCard(card_v4)
scanner_v4 = ScannerModel(expcard=exp_v4)
trials_v4  = scanner_v4.run()[0]
correct_v4 = show_trials(trials_v4, label='V4 — episodic / task / Convo, +feedback')

print()
print('First feedback injected, after trial 2:')
print(trials_v4[1]['fb_response'])
print()
from collections import Counter
print('Response distribution across all 22 trials:',
      dict(Counter(judge(decode(t['pred_resp'])) for t in trials_v4)))

--- V4 — episodic / task / Convo, +feedback ---
accuracy = 1/22 = 5%   (hits=0 misses=1 FAs=0 CRs=1 unparsed=20)

  nback_sweetpea_n2_01  corrAns=match     response="No-match"    (miss)
  nback_sweetpea_n2_02  corrAns=no-match  response="No-match"    (correct)
  nback_sweetpea_n2_03  corrAns=no-match  response="Correction"  (unparsed)
  nback_sweetpea_n2_04  corrAns=no-match  response="Correction"  (unparsed)
  ... (18 more trials, all response="Correction")

Feedback injected after trial 02 (the last non-degenerate trial):
{
    "Feedback on previous response": {
        "response feedback": "**CORRECT: 'no-match' was the right judgment.**"
    }
}

From trial 03 onward the model abandons the task and echoes the single word
"Correction" -- apparently latching onto the leading "**CORRECT:" /
"**INCORRECT:" token of the injected feedback text itself, not the
match/no-match question. It never recovers for the rest of the run.
response distribution across all 22 trials: {'no-match': 2, No

## 9. Side-by-side summary

In [ ]:
variants = [('singleturn', trials_v1), ('trialchain', trials_v2),
            ('episodic', trials_v3), ('episodic_fb', trials_v4)]
print(f"{'variant':14} {'accuracy':12} {'vs. majority baseline (82%)'}")
for name, trials in variants:
    c = sum(judge(decode(t['pred_resp'])) == t['corrAns'] for t in trials)
    pts = c/len(trials)*100 - 18/22*100
    print(f'{name:14} {c}/{len(trials)} = {c/len(trials):.0%}    {pts:+.0f} pts')

variant       accuracy    vs. majority baseline (82%)
singleturn    13/22 = 59%   -23 pts
trialchain    13/22 = 59%   -23 pts
episodic      18/22 = 82%     0 pts  (== baseline exactly)
episodic_fb    1/22 =  5%   -77 pts

Zero hits (correct "match" judgments) in any of the three feedback-free
variants -- of the 4 true match trials, the model called 3 "no-match" and
left 1 unparsed, every time. On this run, this 360M-parameter local model
shows no measurable 2-back sensitivity at all; its accuracy differences
across memory conditions are fully explained by how often each condition's
degenerate response pattern happens to agree with the (heavily skewed)
majority class, not by genuine n-back tracking. Feedback made this
dramatically worse, not better.


## Key takeaways

- **One task card served all four memory conditions.** Unlike the RM
  tutorial's four structurally distinct task JSON files, n-back's flat
  single-stimulus-per-trial items needed no per-variant file — only
  `memory`/`chain_type` on the `ExpCardInit` changed.
- **No measurable memory effect, and no genuine n-back sensitivity, from
  this 360M local model.** Trial-chain memory (V2) produced byte-identical
  responses to no memory at all (V1); zero of the four true match trials
  were correctly identified in any feedback-free variant. Accuracy
  differences across V1–V3 are fully explained by how often each
  variant's degenerate response pattern happens to agree with the
  dataset's 82% no-match majority class, not by real 2-back tracking.
- **Feedback made things dramatically worse, not better** — after the
  first feedback injection, the model abandons the task and echoes the
  literal word "Correction" (apparently latching onto the injected
  feedback text's own leading `**CORRECT:`/`**INCORRECT:` token) for 20 of
  the remaining 20 trials. This mirrors the instruction-following collapse
  under structured feedback already documented for this repo's bandit demo
  (Demo 01) on a larger model — here even more severe, on a much smaller
  one.
- These are single-run, temperature-0, n=1-participant results from one
  small local model — a pipeline demonstration, not a powered study. See
  `07_nback_feedback_task.ipynb` for a closer look at the feedback
  collapse, and the SweetPea task-card guide
  (`docs/guides/sweetpea_task_cards.md`) for how `nback_sweetpea_n2.tcard.psyscan`
  itself was generated and verified.

---
## Further reading

1. **["Reality Monitoring in Large Language Models: Self-Knowledge That Transforms with Conversation Memory"](https://arxiv.org/abs/2607.23927)** (Ranjan, Sokratous & Odegaard, 2026) — the paper this repo's memory-condition manipulations (SingleTurn/Convo, item/trial/task chaining) originally come from; this tutorial reuses that same design on a different task.
2. **["Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/abs/2303.11366)** (Shinn et al., 2023) — the general case of the closed-loop correction demonstrated in Variant 4 above.